In [58]:
# Knowledge Base represented as a list of triples
knowledge_base = [
    ("Animal", "has", "skin"),
    ("Bird", "is-a", "Animal"),
    ("Bird", "can", "fly"),
    ("Penguin", "is-a", "Bird"),
    ("Penguin", "cannot", "fly"),
    ("Fish", "is-a", "Animal"),
    ("Shark", "is-a", "Fish"),
    
]

In [59]:
def findParent(subject:str):
    temp=[]

    for line in knowledge_base:
        if line[0]==subject and line[1]=="is-a":
            temp.append(line[2]) 

    return temp

def findInheritance(subject:str,printParents=False):
    toSearch=findParent(subject)
    parents=toSearch

    for s in toSearch:
        for p in findInheritance(s,printParents=False):
                parents.append(p)

    if printParents:
        print(subject,end=" ")

        for p in parents:
            print("-> ",p,end=" ")

    return parents


In [60]:
list=findInheritance("Penguin",True)

Penguin ->  Bird ->  Animal 

In [61]:
def localQuery(subject,relation):
    about_subject=[line for line in knowledge_base if line[0]==subject]

    for line in about_subject:
        if line[1]==relation:
            return "no exception",line[2]
        
    for line in about_subject:
        if line[1]==relation+"not":
            return "exception",line[2]

    

    return None,None

def query(subject,relation,printAnswer=False):
    status,answer=localQuery(subject,relation)

    if printAnswer:
        if answer:
            if status=="no exception":
                print(f"{subject} {relation} {answer}.   {status}")
            else:
                print(f"{subject} {relation}not {answer}.   {status}")

            return status,answer

    parents=findInheritance(subject)

    for  p in parents:
        status,answer=localQuery(p,relation)

    if printAnswer:
        if answer:
            if status=="no exception":
                print(f"{subject} {relation} {answer}.   {status}")
            else:
                print(f"{subject} {relation}not {answer}.   {status}")

        else:
            answer="unknown"
            status="not found"
            print(f"{subject} {relation} {answer}.   {status}")

    return status,answer



In [62]:
status,ans=query("Penguin","has",printAnswer=True)

Penguin has skin.   no exception


In [63]:
ans=query("Penguin","can",printAnswer=True)

Penguin cannot fly.   exception


In [64]:
status,ans=query("fish","can",printAnswer=True)

fish can unknown.   not found


In [ ]:
knowledge_base.append(("Ostrich", "is-a", "Bird"))
knowledge_base.append(("Ostrich", "cannot", "fly"))
knowledge_base.append(("Ostrich", "can", "run fast"))

In [66]:
status,ans=query("Ostrich","has",printAnswer=True)
status,ans=query("Ostrich","can",printAnswer=True)

Ostrich has skin.   no exception
Ostrich cannot fly.   exception


In [67]:
frames = {
    "animal":  {"ako": None,     "has": "skin"},
    "bird":    {"ako": "animal", "can": "fly",  "has": "feathers"},
    "fish":    {"ako": "animal", "can": "swim", "has": "scales"},
    "penguin": {"ako": "bird",   "cannot": "fly", "can": "swim"},
    "ostrich": {"ako": "bird",   "cannot": "fly", "can": "run fast"},
    "shark":   {"ako": "fish"},
}

In [68]:


def frame_chain(concept):

    chain = [concept]
    current = concept
    while frames[current]["ako"] is not None:
        current = frames[current]["ako"]
        chain.append(current)
    return chain


def frame_query(concept, slot, value=None):

    chain = frame_chain(concept)
    for level in chain:
        f = frames[level]
        if slot == "can" and "cannot" in f and (value is None or f["cannot"] == value):
            print(f"{concept} cannot {f['cannot']}")
            return f"cannot {f['cannot']}"
        if slot in f and (value is None or f[slot] == value):
            print(f"{concept} {slot} {f[slot]}")
            return f"{slot} {f[slot]}"
    return "unknown"




In [69]:
print( " -> ".join(frame_chain("penguin")))

penguin -> bird -> animal


In [70]:
frame_query("penguin", "can")
frame_query("shark", "has")
frame_query("ostrich", "has")
frame_query("animal", "has")


penguin cannot fly
shark has scales
ostrich has feathers
animal has skin


'has skin'